# AI-Based Detection of Spam and Malicious URLs

A machine learning notebook for binary malicious URL classification using lexical feature engineering, Logistic Regression, Random Forest, and XGBoost.

## What this notebook demonstrates

- Loading a labeled URL dataset with benign and malicious categories.
- Transforming multi-class URL labels into a binary cybersecurity detection task.
- Extracting lightweight lexical URL features without visiting web pages.
- Training Logistic Regression, Random Forest, and XGBoost classifiers.
- Evaluating accuracy, precision, recall, F1-score, confusion matrix, ROC curve, and Precision-Recall curve.
- Interpreting feature importance for the best-performing tree-based model.

## Local dataset path

Place the dataset locally as:

```text
data/malicious_phish.csv
```

The expected columns are:

```text
url,type
```



### Import Libraries and Load Dataset
Importing essential Python libraries for data manipulation, visualization, and machine learning models. We also load the dataset to verify its structure and content.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Import utilities for data splitting and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Import Machine Learning Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Load the dataset (Ensure the file path is correct)
df = pd.read_csv("data/malicious_phish.csv")

# Display dataset dimensions and columns
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

# Show the first few rows
df.head()


### Exploratory Data Analysis (EDA)
Here, we analyze the distribution of the target classes to understand the dataset balance. We also check for missing values and verify data types to ensure data quality before processing.


In [ ]:
# Visualize the distribution of original multi-class labels before binary transformation
df['type'].value_counts().plot(kind='bar', color=['green','red','orange','purple'])
plt.title("Class Distribution (Original Multi-Class Labels)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

# Display General Data Information
print("\nData Information")
df.info()

# Check for Missing Values
print("\nMissing Values")
print(df.isna().sum())


### Label Encoding
In this step, we simplify the problem into a Binary Classification task. The original dataset contains multiple types of attacks (Phishing, Malware, Defacement). We group all these malicious types under class 1 and keep benign URLs as class 0.

Mapping:
 0 : Benign
 1 : Malicious


In [ ]:
# Create a new column label for binary classification
df['label'] = df['type'].apply(lambda x: 0 if x == 'benign' else 1)

# Verify the conversion by displaying sample rows
print("Sample Data after Encoding")
print(df[['url', 'type', 'label']].head())

# Display the count of each class
print("\nBinary Class Counts ")
print(df['label'].value_counts())

# Visualize the Binary Class Distribution
df['label'].value_counts().plot(kind='bar')
plt.title("Binary Class Distribution (Benign vs Malicious)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks([0, 1], ['Benign', 'Malicious'], rotation=0)
plt.show()


### Feature Engineering
In this critical step, we extract numerical features directly from the URL strings. These lexical features help the model identify suspicious patterns often used by attackers, such as:
* URL Length & Special Characters: Malicious URLs are often longer and contain more symbols for obfuscation.
* IP Address: Using raw IP addresses instead of domain names is a common malware indicator.
* Suspicious Keywords: Words like "login", "verify", or "secure" are frequently used in social engineering attacks.


In [ ]:
def extract_features(url: str):
    # Convert URL to string to ensure compatibility
    url = str(url)
    features = {}

    # Structural Features
    features['url_length']   = len(url)
    features['num_digits']   = sum(c.isdigit() for c in url)
    features['num_special']  = sum(not c.isalnum() for c in url)
    features['num_dots']     = url.count('.')

    # Pattern Features

    # Check if the URL is using a raw IP address
    features['has_ip']       = 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0

    # Check protocol
    features['has_https']    = 1 if url.lower().startswith("https") else 0

    # Keyword Features
    features['has_login']    = 1 if "login" in url.lower() else 0
    features['has_verify']   = 1 if "verify" in url.lower() else 0
    features['has_secure']   = 1 if "secure" in url.lower() else 0

    return features

# Apply feature extraction to the entire dataset
feature_rows = df['url'].apply(extract_features)

# Create the Feature Matrix
X = pd.DataFrame(list(feature_rows))
y = df['label']

# Display the first 5 rows
X.head()


### Data Splitting and Standardization
We split the dataset into 80% training and 20% testing sets.
*Stratification: We use `stratify=y` to maintain the same ratio of benign/malicious URLs in both sets.
* Scaling: We apply `StandardScaler ` to normalize features. This is specifically required for the Logistic Regression model to ensure efficient convergence, while tree-based models can handle raw features.


In [ ]:
# Split the data into Training and Testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()

# Fit on training set
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


### Model Training, Evaluation, and Visualization
In this final stage, we train three different classifiers: Logistic Regression, Random Forest, and XGBoost. We compare their performance using standard metrics (Accuracy, Precision, Recall, F1-Score) and visualize the Confusion Matrix for the bestperforming model.


In [ ]:
# Define models with specific hyperparameters
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        n_jobs=-1,
        random_state=42
    )
}

results = []

# Variables to track the best model
best_model_name = ""
best_accuracy = 0.0
best_y_pred = []

print("Training Models...\n")

for name, model in models.items():
    print(f" {name} ")

    # Apply scaling ONLY for Logistic Regression
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        # Tree-based models (RF, XGB) do not require scaling
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Calculate Evaluation Metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1)
    rec  = recall_score(y_test, y_pred, pos_label=1)
    f1   = f1_score(y_test, y_pred, pos_label=1)

    results.append([name, acc, prec, rec, f1])

    print(classification_report(y_test, y_pred, target_names=['benign', 'malicious']))

    # Logic to automatically select the best model
    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = name
        best_y_pred = y_pred

# Display Final Comparison Table
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1"])
print("\n Final Comparison")
print(results_df)
print(f"\nThe Best Model is: {best_model_name} with Accuracy: {best_accuracy:.4f}")

# Plot Confusion Matrix for the Best Model
cm = confusion_matrix(y_test, best_y_pred)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

# Set axis labels
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Benign', 'Malicious'])
ax.set_yticklabels(['Benign', 'Malicious'])

# Annotate confusion matrix with counts
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title(f"Confusion Matrix - {best_model_name}\n(Accuracy: {best_accuracy:.4f})")

plt.tight_layout()
plt.show()


### Feature Importance Analysis
Understanding why a model classifies a URL as malicious is crucial for cybersecurity. In this step, we extract and visualize the Feature Importance from the best-performing model. This highlights which lexical features contribute most to the detection.


In [ ]:
# Retrieve the best model object from the dictionary using its name
best_model = models[best_model_name]

# Check if the model supports feature importance (Tree-based models do)
if hasattr(best_model, 'feature_importances_'):

    # Define feature names (ensure they match the order in extraction)
    feature_names = ['url_length', 'num_digits', 'num_special', 'num_dots', 'has_ip', 'has_https', 'has_login', 'has_verify', 'has_secure']

    # Extract importances
    importances = best_model.feature_importances_

    # Create a Series for easier plotting
    forest_importances = pd.Series(importances, index=feature_names).sort_values(ascending=False)

    # Plot the Feature Importance
    plt.figure(figsize=(8, 6))
    sns.barplot(x=forest_importances.values, y=forest_importances.index, palette="viridis")

    plt.title(f"Feature Importance - {best_model_name}")
    plt.xlabel("Importance Score")
    plt.ylabel("Features")
    plt.tight_layout()
    plt.show()

else:
    print(f"The best model ({best_model_name}) is a linear model and does not support 'feature_importances_' in this format.")


### Model Accuracy Comparison
Finally, we visualize the accuracy of all trained models side-by-side. This bar chart clearly illustrates the performance gap between linear models (Logistic Regression) and tree-based ensemble models (Random Forest & XGBoost), supporting the findings discussed in the report.


In [ ]:
# Extract data for plotting from the results dataframe
models_names = results_df['Model']
accuracies = results_df['Accuracy']

plt.figure(figsize=(8, 5))

# Draw bars with distinct colors
bars = plt.bar(models_names, accuracies, color=['#bdc3c7', '#2ecc71', '#3498db'])

# Add accuracy numbers above each bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, round(yval, 3),
             ha='center', va='bottom', fontweight='bold')

# Enhance chart appearance
plt.ylim(0.7, 1.0)
plt.title("Accuracy Comparison Between Models", fontsize=14)
plt.ylabel("Accuracy Score", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# Calculate the scores according to the model type
if best_model_name == "Logistic Regression":
    y_scores = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    y_scores = best_model.predict_proba(X_test)[:, 1]

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve - {best_model_name}")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# Precision–Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_scores)
pr_auc = average_precision_score(y_test, y_scores)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"PR curve (AP = {pr_auc:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall Curve - {best_model_name}")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.show()


### Conclusion and Key Findings

In this project, we successfully built and evaluated a machine learning pipeline for detecting malicious URLs based on lexical features.

Summary of Achievements:
* Data Handling: We processed a dataset of over 650,000 URLs, handling class imbalance and extracting meaningful features like URL length, special character count, and IP usage.
* Model Performance: We trained and compared three models:
    1.  Logistic Regression: Served as a baseline with reasonable precision but lower recall (~74.9% Accuracy).
    2.  Random Forest: Achieved the highest overall performance (~87.6% Accuracy), demonstrating robustness against obfuscated URLs.
    3.  XGBoost: Performed competitively (~86.6% Accuracy), very close to Random Forest.
* Key Insights: Feature importance analysis revealed that structural anomalies (e.g., count of special characters like `@`, `.`, `-`) are the strongest indicators of malicious intent.

Final Verdict:
Based on the experimental results, Random Forest is recommended as the optimal model for this specific task due to its superior accuracy and balanced precision-recall trade-off.
